In [1]:
# ===============================
# CORE PYTHON
# ===============================
import os
import time
import threading
from queue import Queue
import platform

# ===============================
# NUMPY & IMAGE
# ===============================
import numpy as np
from PIL import Image

# ===============================
# COMPUTER VISION
# ===============================
import cv2
import torch
from ultralytics import YOLO

# ===============================
# DEPTH ESTIMATION (MiDaS)
# ===============================
import torchvision.transforms as transforms

# ===============================
# FACE + EMOTION
# ===============================
from deepface import DeepFace

# ===============================
# SPEECH RECOGNITION
# ===============================
import speech_recognition as sr

# ===============================
# TEXT TO SPEECH (OFFLINE)
# ===============================
import pyttsx3

# ===============================
# GPS / LOCATION
# ===============================
import geocoder

# ===============================
# EMERGENCY SMS / CALL
# ===============================
from twilio.rest import Client

# ===============================
# GUI (Optional Later)
# ===============================
import customtkinter as ctk

In [2]:
# ===============================
# GLOBAL CONFIG
# ===============================

CAMERA_INDEX = 0
YOLO_MODEL_PATH = "yolov8n.pt"

WAKE_WORDS = ["Synthea", "emergency"]

NAVIGATION_MODE = False
RUNNING = True

# Distance thresholds (approx meters)
DANGER_DISTANCE = 1.0
SAFE_DISTANCE = 2.0

In [3]:
# ===============================
# EVENT QUEUE (Brain Communication)
# ===============================

event_queue = Queue()

In [4]:
# ===============================
# TTS ENGINE
# ===============================

engine = pyttsx3.init()
engine.setProperty('rate', 170)  # speed adjust
engine.setProperty('volume', 1)

def speak(text):
    print("Synthea:", text)
    engine.say(text)
    engine.runAndWait()

In [5]:
# ===============================
# LOAD YOLO MODEL
# ===============================

device = "cuda" if torch.cuda.is_available() else "cpu"

yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_model.to(device)

print("YOLO Loaded on:", device)
# ===============================
# LOAD YOLO MODEL (REQUIRED)
# ===============================

from ultralytics import YOLO
import torch

YOLO_MODEL_PATH = "yolov8n.pt"

device = "cuda" if torch.cuda.is_available() else "cpu"

yolo_model = YOLO(YOLO_MODEL_PATH)
yolo_model.to(device)

print("YOLO Loaded on:", device)

YOLO Loaded on: cpu
YOLO Loaded on: cpu


In [6]:
# ===============================
# SYNTHEA - CAMERA INIT
# ===============================

ANDROID_INDEX = 1  # jo tera correct index hai

cap = cv2.VideoCapture(ANDROID_INDEX)

if not cap.isOpened():
    raise Exception("Camera not detected ❌")

# 🔥 YAHI PE RESOLUTION SET KARNA HAI
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
cap.set(cv2.CAP_PROP_FPS, 24)

print("Camera Connected ✅")

2026-03-17 22:42:04.707 python[11760:307697] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Camera Connected ✅


# ===============================
# SYNTHEA - STABLE OBSTACLE SYSTEM
# ===============================


In [8]:
SPEAK_INTERVAL = 5
last_spoken_time = 0

def detect_obstacles(frame):
    global last_spoken_time

    results = yolo_model(frame, verbose=False)
    h, w, _ = frame.shape

    detected_objects = []

    for box in results[0].boxes:
        cls = int(box.cls[0])
        label = yolo_model.names[cls]

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        area = (x2 - x1) * (y2 - y1)

        if area < 2500:
            continue

        center_x = (x1 + x2) // 2
        angle = ((center_x / w) - 0.5) * 60
        angle = round(angle)

        detected_objects.append(f"{label} at {angle} degrees")

        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, f"{label} ({angle}°)",
                    (x1,y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    current_time = time.time()

    if current_time - last_spoken_time > SPEAK_INTERVAL:

        if len(detected_objects) == 0:
            speak("Path is clear.")

        elif len(detected_objects) <= 3:
            speak(". ".join(detected_objects))

        else:
            speak("Many obstacles ahead. Please choose a different path.")

        last_spoken_time = current_time

    return frame, []

In [10]:
# ===============================
# SYNTHEA - TEXT TO SPEECH SETUP
# ===============================

import pyttsx3

engine = pyttsx3.init()
engine.setProperty('rate', 170)
engine.setProperty('volume', 1)

def speak(text):
    print("Synthea:", text)
    engine.say(text)
    engine.runAndWait()

# ===============================
# SYNTHEA - SMART OBSTACLE FILTER
# ===============================

last_spoken_time = 0
SPEAK_INTERVAL = 3

# Only major physical obstacles
OBSTACLE_CLASSES = {
    "person", "car", "truck", "bus",
    "motorcycle", "bicycle",
    "chair", "sofa", "couch",
    "dining table", "bench", "bed",
    "suitcase", "refrigerator"
}

def detect_obstacles(frame):
    global last_spoken_time

    results = yolo_model(frame, verbose=False)
    frame_height, frame_width, _ = frame.shape

    obstacles = []

    for box in results[0].boxes:
        cls = int(box.cls[0])
        label = yolo_model.names[cls]

        # 🔥 Ignore small useless detections
        if label not in OBSTACLE_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        box_area = (x2 - x1) * (y2 - y1)

        # 🔥 Ignore very small objects (far away or tiny)
        if box_area < 15000:
            continue

        center_x = (x1 + x2) // 2

        # Direction logic
        if center_x < frame_width / 3:
            direction = "left"
        elif center_x > 2 * frame_width / 3:
            direction = "right"
        else:
            direction = "center"

        obstacles.append((label, direction))

        # Draw bounding box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, f"{label} ({direction})",
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0,255,0), 2)

        # Voice alert only if big obstacle in center
        if direction == "center":
            current_time = time.time()
            if current_time - last_spoken_time > SPEAK_INTERVAL:
                speak(f"{label} ahead. Please stop.")
                last_spoken_time = current_time

    return frame, obstacles

In [ ]:
# ===============================
# SYNTHEA - CONTINUOUS NAVIGATION
# ===============================

print("Synthea Navigation Mode Started...")
speak("Navigation mode activated.")

while True:
    ret, frame = cap.read()

    if not ret:
        print("Frame capture failed")
        break

    # ❌ NO resize here (important)

    # Detect obstacles
    frame, obstacles = detect_obstacles(frame)

    # Show live feed
    cv2.imshow("Synthea - Navigation Mode", frame)

    # Exit on Q
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print("Navigation mode stopped.")
speak("Navigation mode stopped.")

Synthea Navigation Mode Started...
Synthea: Navigation mode activated.
Synthea: Path is clear.
Synthea: Many obstacles ahead. Please choose a different path.
Synthea: traffic light at 15 degrees
Synthea: Many obstacles ahead. Please choose a different path.
Synthea: person at -16 degrees
Synthea: person at -17 degrees. person at 11 degrees. person at 9 degrees
Synthea: Path is clear.
Synthea: person at -2 degrees
Synthea: Many obstacles ahead. Please choose a different path.
Synthea: Path is clear.
Synthea: Path is clear.
Synthea: Path is clear.
Synthea: Path is clear.
Synthea: Path is clear.
Synthea: vase at -14 degrees
Synthea: keyboard at -7 degrees
Synthea: Many obstacles ahead. Please choose a different path.
Synthea: keyboard at 3 degrees. person at -27 degrees. cup at -8 degrees
Synthea: dining table at 0 degrees
Synthea: Path is clear.
